In [ ]:
import json
import csv
import os # Import os module for path handling

# --- Configuration: Define file paths for Colab ---
# Assumes files are in a 'data' folder within your main Colab directory '/content/'
colab_data_folder = '/content/data'
# *** IMPORTANT: Change these filenames if yours are different ***
jsonl_file_name = 'your_file.jsonl'
csv_input_file_name = 'input.csv'
csv_output_file_name = 'output.csv'

# Construct full paths
jsonl_file_path = os.path.join(colab_data_folder, jsonl_file_name)
csv_input_path = os.path.join(colab_data_folder, csv_input_file_name)
csv_output_path = os.path.join(colab_data_folder, csv_output_file_name)

# --- 1. Ensure the data folder exists (optional, good practice) ---
# You might need to create this folder manually in Colab's file explorer
# or uncomment the line below to create it programmatically.
# os.makedirs(colab_data_folder, exist_ok=True)
print(f"Expecting JSONL file at: {jsonl_file_path}")
print(f"Expecting CSV input file at: {csv_input_path}")
print(f"Output CSV will be saved to: {csv_output_path}")


# --- 2. Read and Process JSONL File ---
jsonl_extracted_data = []
jsonl_read_successful = False # Flag to track success
try:
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        print(f"\nReading JSONL file: {jsonl_file_path}...")
        for line_num, line in enumerate(f, 1):
            if line.strip(): # Check if line is not just whitespace
                try:
                    # Decode escaped characters like \n in keys/values if necessary
                    # It seems your keys have literal newlines, which json.loads handles directly
                    # decoded_line = line.encode().decode('unicode_escape') # Usually not needed if keys are literal
                    data = json.loads(line)
                    # Extract the required fields using the exact keys
                    ability = data.get("能力\nABILITY", "") # Use .get for safety if key might be missing
                    intention_index = data.get("序号\nINDEX", "")
                    if "能力\nABILITY" not in data:
                         print(f"Warning: Key '能力\\nABILITY' not found in JSONL line {line_num}.")
                    if "序号\nINDEX" not in data:
                         print(f"Warning: Key '序号\\nINDEX' not found in JSONL line {line_num}.")
                    jsonl_extracted_data.append((ability, intention_index))
                except json.JSONDecodeError as e:
                    print(f"ERROR: Skipping invalid JSON line #{line_num}: {line.strip()} - Error: {e}")
                except KeyError as e:
                    print(f"ERROR: Missing key {e} in JSONL line #{line_num}: {line.strip()}")
                except Exception as e:
                     print(f"ERROR: An unexpected error occurred processing JSONL line #{line_num}: {line.strip()} - Error: {e}")
        print(f"Successfully processed {len(jsonl_extracted_data)} lines from JSONL file.")
        jsonl_read_successful = True # Mark as successful

except FileNotFoundError:
    print(f"ERROR: JSONL file not found at {jsonl_file_path}. Please ensure it exists in the 'data' folder.")
except Exception as e:
    print(f"ERROR: An critical error occurred reading the JSONL file: {e}")

# --- 3. Read CSV, Merge Data, and Write Output CSV ---
if jsonl_read_successful and jsonl_extracted_data: # Proceed only if JSONL reading was successful and data was extracted
    print(f"\nReading CSV file: {csv_input_path} and writing to {csv_output_path}...")
    rows_written = 0
    try:
        with open(csv_input_path, 'r', encoding='utf-8', newline='') as infile, \
             open(csv_output_path, 'w', encoding='utf-8', newline='') as outfile:

            csv_reader = csv.reader(infile)
            csv_writer = csv.writer(outfile)

            try:
                # Read header from input CSV
                header = next(csv_reader)
                # Define new header for output CSV
                new_header = header + ['Ability', 'Intention_Index']
                # Write the new header to the output file
                csv_writer.writerow(new_header)
                rows_written += 1 # Count header row

                # Iterate through input CSV rows and merge with extracted JSONL data
                for i, row in enumerate(csv_reader):
                     # Check if there is corresponding data from the JSONL file
                     if i < len(jsonl_extracted_data):
                         # Get corresponding ability and intention_index
                         ability, intention_index = jsonl_extracted_data[i]
                         # Append the new data to the current CSV row
                         row.extend([ability, str(intention_index)]) # Ensure index is string for CSV
                         # Write the merged row to the output file
                         csv_writer.writerow(row)
                         rows_written += 1
                     else:
                         # Handle cases where CSV has more data rows than JSONL
                         print(f"Warning: CSV data row {i+1} (line {i+2} in file) has no corresponding JSONL data. This row will not be included in the output.")
                         # If you want to include these rows with empty placeholders, uncomment below:
                         # row.extend(["", ""]) # Add empty placeholders
                         # csv_writer.writerow(row)
                         # rows_written += 1

                print(f"\n--- Merge complete. Successfully wrote {rows_written} rows (including header) to {csv_output_path} ---")

            except StopIteration:
                print("ERROR: Input CSV file appears to be empty or only contains a header. No data rows processed.")
            except Exception as e:
                print(f"ERROR: An error occurred during CSV processing: {e}")

    except FileNotFoundError:
        print(f"ERROR: Input CSV file not found at {csv_input_path}. Please ensure it exists in the 'data' folder.")
    except Exception as e:
        print(f"ERROR: An critical error occurred opening or writing CSV files: {e}")

elif not jsonl_read_successful:
     print("\nSkipping CSV processing due to errors reading the JSONL file.")
elif not jsonl_extracted_data:
     print("\nSkipping CSV processing because no data was successfully extracted from the JSONL file.")
else:
     print("\nSkipping CSV processing due to an unknown issue.")



Expecting JSONL file at: /content/data/your_file.jsonl
Expecting CSV input file at: /content/data/input.csv
Output CSV will be saved to: /content/data/output.csv

Reading JSONL file: /content/data/your_file.jsonl...
Successfully processed 2860 lines from JSONL file.

Reading CSV file: /content/data/input.csv and writing to /content/data/output.csv...

--- Merge complete. Successfully wrote 2861 rows (including header) to /content/data/output.csv ---
